# Clustering: recovering hepatitis C categories without labels

This notebook walks through the clustering analysis in the order the
coursework took it. It calls the same functions as
`analysis/m02_clustering.py`, and the write-up with every table and figure is
[docs/02-clustering.md](../docs/02-clustering.md).

The HCV data (Lichtinghagen et al. 2020) holds ten laboratory values for 615
subjects with a diagnostic category for each: blood donor, hepatitis, fibrosis
or cirrhosis. The category is withheld from every clustering and used
afterwards to score agreement, with the adjusted Rand index and adjusted
mutual information, beside three internal indices computed from the clustered
points alone.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Every output directory is redirected to a temporary location before the
# pipeline modules are imported, so this notebook writes nothing into
# results/, figures/ or data/processed/. The committed tables are read from
# results/ directly where the notebook quotes them.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
_scratch = tempfile.mkdtemp()
for _name in ("ML_METHODS_RESULTS", "ML_METHODS_FIGURES", "ML_METHODS_PROCESSED"):
    os.environ[_name] = _scratch
sys.path.insert(0, str(ROOT))

import numpy
import pandas

from src import config, data, evaluate, splits

RESULTS = ROOT / "results"
pandas.set_option("display.width", 120)
pandas.set_option("display.max_columns", 20)


def recorded(prefix=""):
    """The committed metrics record, as a dictionary of strings."""
    table = pandas.read_csv(RESULTS / "metrics.csv", dtype=str)
    return {row.key: row.value for row in table.itertuples()
            if row.key.startswith(prefix)}

## Data and cleaning

`data.load_hcv` removes the 7 subjects labeled as suspect donors, renames
`CGT` to `GGT`, and imputes the 31 missing cells from the five nearest
subjects on the assays each subject does hold, on the standardized scale and
without reading the category. The imputed rows are flagged. The counts say
which categories a complete-case rule would have thinned.

In [2]:
frame, counts = data.load_hcv()
pandas.Series(counts)

rows_released            615
released_0               533
released_3                30
released_1                24
released_2                21
released_0s                7
rows_suspect_donor         7
rows_incomplete           26
cells_imputed             31
cells_total             6080
cells_imputed_ALB          1
cells_imputed_ALP         18
cells_imputed_ALT          1
cells_imputed_CHOL        10
cells_imputed_PROT         1
rows_analyzed            608
rows_complete            582
categories_analyzed        4
analyzed_blood_donor     533
imputed_blood_donor        7
analyzed_cirrhosis        30
imputed_cirrhosis          6
analyzed_fibrosis         21
imputed_fibrosis           9
analyzed_hepatitis        24
imputed_hepatitis          4
dtype: int64

In [3]:
features = list(config.HCV_FEATURES)
frame[features + ["imputed", "category_label"]].head()

,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,imputed,category_label
0,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,False,Blood Donor
1,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,False,Blood Donor
2,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,False,Blood Donor
3,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,False,Blood Donor
4,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,False,Blood Donor


## Inputs

The ten assays differ in variance by four orders of magnitude, and two of
them, GGT and creatinine, hold two thirds of the total. The panel is
standardized before any distance is computed. That decision was made on the
scales and not on the agreement, and both agreements are recorded below so the
cost of the decision is visible.

In [4]:
from sklearn.preprocessing import StandardScaler

raw = frame[features].to_numpy(dtype=float)
scaled = StandardScaler().fit_transform(raw)
reference = frame["category_index"].to_numpy()

share = pandas.Series(raw.var(axis=0, ddof=1), index=features)
(share / share.sum()).sort_values(ascending=False).round(4)

GGT     0.3480
CREA    0.3187
AST     0.1372
ALP     0.0801
ALT     0.0580
BIL     0.0504
ALB     0.0038
PROT    0.0031
CHE     0.0006
CHOL    0.0002
dtype: float64

![Each assay's share of the total unstandardized variance.](../figures/fig04_hcv_variance.png)

## Training: three methods at four clusters

k-means at k = 4, DBSCAN at the neighborhood parameters the pipeline selected
by Calinski-Harabasz from 45 admitted configurations, and agglomerative
clustering with complete linkage at k = 4. Each is scored on the three
internal indices and, afterwards, on agreement with the withheld category.

In [5]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from analysis import m02_clustering as m02

record = recorded("m02.")
eps = float(record["m02.dbscan.eps"])
min_samples = int(record["m02.dbscan.min_samples"])

labels = {
    "k-means": m02.fit_kmeans(scaled, config.HCV_K).labels_,
    "DBSCAN, eps {} min_samples {}".format(eps, min_samples):
        DBSCAN(eps=eps, min_samples=min_samples).fit(scaled).labels_,
    "agglomerative, complete linkage":
        AgglomerativeClustering(n_clusters=config.HCV_K, linkage="complete").fit_predict(scaled),
}
rows = []
for name, assigned in labels.items():
    row = {"method": name}
    row.update(evaluate.clustering(scaled, assigned))
    row.update(evaluate.agreement(reference, assigned))
    rows.append(row)
pandas.DataFrame(rows).set_index("method").T

method,k-means,"DBSCAN, eps 4.0 min_samples 3","agglomerative, complete linkage"
clusters,4.000000,3.000000,4.000000
unassigned,0.000000,12.000000,0.000000
assigned_fraction,1.000000,0.980263,1.000000
smallest_cluster,3.000000,3.000000,1.000000
calinski_harabasz,102.538373,45.451888,63.823983
silhouette,0.165097,0.639167,0.663857
davies_bouldin,1.508694,0.431422,0.702578
adjusted_rand,0.131940,0.310340,0.249161
adjusted_mutual_information,0.196556,0.204557,0.193593
homogeneity,0.277015,0.142778,0.125288


## Results

Calinski-Harabasz ranks k-means first by a factor of two and agreement with
the labels ranks it last. The index is the ratio of between-cluster to
within-cluster dispersion, which is the quantity k-means minimizes, so on the
same data it will usually score higher on that index than a method optimizing
something else. Silhouette and Davies-Bouldin, also computed without labels,
rank k-means last.

![Calinski-Harabasz and adjusted Rand index as the number of clusters varies.](../figures/fig05_hcv_cluster_sweep.png)

In [6]:
pandas.crosstab(labels["k-means"], frame["category_label"],
                rownames=["cluster"], colnames=["category"])

category,Blood Donor,Cirrhosis,Fibrosis,Hepatitis
cluster,,,,
0,257,4,12,13
1,276,3,7,7
2,0,3,0,0
3,0,20,2,4


Two of the four clusters partition the donors, and the 45 hepatitis and
fibrosis cases sit inside them with no cluster holding a majority of either.
Cluster 3 is a cirrhosis group.

![The 608 subjects in the first two principal components, by cluster and by category.](../figures/fig06_hcv_projection.png)

### Where the recovery is strong

The coursework also compared a two-cluster solution against cirrhosis held
apart from every other category.

In [7]:
two = m02.fit_kmeans(scaled, 2).labels_
cirrhosis = (frame["category_index"] == 3).astype(int)
print(pandas.Series(evaluate.agreement(cirrhosis, two)).round(4))
pandas.crosstab(two, cirrhosis.map({0: "other", 1: "cirrhosis"}),
                rownames=["cluster"], colnames=["category"])

adjusted_rand                  0.8152
adjusted_mutual_information    0.6653
homogeneity                    0.6245
completeness                   0.7158
dtype: float64


category,cirrhosis,other
cluster,,
0,7,576
1,23,2


Twenty-three of the 30 cirrhosis cases fall in a cluster of 25, an adjusted
Rand index of 0.815. The cluster's median albumin is 32.0 against 42.2,
bilirubin 40.0 against 7.1 and cholinesterase 2.47 against 8.39, which is the
laboratory picture of failing synthetic liver function.

![Median and quartiles of each assay under the two-cluster solution and under the cirrhosis label.](../figures/fig07_hcv_group_profile.png)

### Sensitivity to the imputed rows and to the seed

The pipeline refitted both solutions on the 582 complete cases and at ten
seeds.

In [8]:
sensitivity = pandas.read_csv(RESULTS / "m02_imputation_sensitivity.csv")
sensitivity[["comparison", "rows_all", "rows_complete", "adjusted_rand_all", "adjusted_rand_complete"]]

,comparison,rows_all,rows_complete,adjusted_rand_all,adjusted_rand_complete
0,"k-means, k=4, against four categories",608,582,0.131940,0.136662
1,"k-means, k=2, against cirrhosis",608,582,0.815224,0.906254


In [9]:
stability = pandas.read_csv(RESULTS / "m02_seed_stability.csv")
stability[["adjusted_rand_k4", "adjusted_rand_k2_vs_cirrhosis"]].agg(["min", "max"])

,adjusted_rand_k4,adjusted_rand_k2_vs_cirrhosis
min,0.087402,0.742818
max,0.144646,0.815224


## Discussion

Cirrhosis is separable from a routine laboratory panel without labels, at an
adjusted Rand index between 0.74 and 0.91 depending on which cases are counted
and where the initialization lands. On the complete cases the two-cluster
agreement rises to 0.906, and the difference is the six cirrhosis cases that
were missing an assay, five of which land with the donors once their missing
value is filled in from their neighbors. Hepatitis and fibrosis are not
separable at this sample size by these three methods; their subjects differ
from healthy donors by less than the donors differ among themselves.

The internal index the coursework used to pick a method would have picked the
one that agrees with the labels least. The write-up sets out the remaining
limitations: the 533 to 75 imbalance that decides where clusters form, the
DBSCAN selection on the evaluation data, the single-subject clusters complete
linkage produces, and the ordering among the four categories that every
measure here ignores.